In [ ]:
!pip install pandas scikit-learn joblib pymongo transformers rouge-score pypdf openai


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 24.9 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=6a2c882080ad62592b130940d5e4ecfef1c4bce7eba611aaa032a063ca119daf
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/archive (5).zip"
extract_path = "/mnt/data/student_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

os.listdir(extract_path)


['student-por.csv', 'student.txt', 'student-merge.R', 'student-mat.csv']

In [ ]:
import pandas as pd

df = pd.read_csv("/mnt/data/student_dataset/student-mat.csv", sep=";")
df.head()


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [ ]:
df = df[["studytime", "G3", "absences"]]

df.columns = ["time_spent", "quiz_score", "absences"]
df.head()


,time_spent,quiz_score,absences
0,2,6,6
1,2,6,4
2,2,10,10
3,3,15,2
4,2,10,4


In [ ]:
# Convert studytime (1–4) → minutes
df["time_spent"] = df["time_spent"] * 15

# Convert grades (0–20) → percentage
df["quiz_score"] = df["quiz_score"] * 5

# Create confidence from absences (inverse relation)
df["confidence"] = 5 - (df["absences"] / df["absences"].max()) * 4
df["confidence"] = df["confidence"].round().clip(1,5)

df.head()


,time_spent,quiz_score,absences,confidence
0,30,30,6,5.0
1,30,30,4,5.0
2,30,50,10,4.0
3,45,75,2,5.0
4,30,50,4,5.0


In [ ]:
def assign_level(score):
    if score < 50:
        return "Beginner"
    elif score < 75:
        return "Intermediate"
    else:
        return "Advanced"

df["level"] = df["quiz_score"].apply(assign_level)
df.head()


,time_spent,quiz_score,absences,confidence,level
0,30,30,6,5.0,Beginner
1,30,30,4,5.0,Beginner
2,30,50,10,4.0,Intermediate
3,45,75,2,5.0,Advanced
4,30,50,4,5.0,Intermediate


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["level"] = le.fit_transform(df["level"])

df["level"].value_counts()


,count
level,
2,192
1,130
0,73


In [ ]:
from sklearn.model_selection import train_test_split

X = df[["confidence", "quiz_score", "time_spent"]]
y = df["level"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        27
           2       1.00      1.00      1.00        32

    accuracy                           1.00        79
   macro avg       1.00      1.00      1.00        79
weighted avg       1.00      1.00      1.00        79



In [ ]:
sample_user = [[3, 65, 30]]  # confidence, quiz_score, time_spent

pred = model.predict(sample_user)
print("Predicted Level:", le.inverse_transform(pred))


Predicted Level: ['Intermediate']


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [ ]:
import joblib

# Save model
joblib.dump(model, "/content/drive/MyDrive/personalization_model.pkl")

# Save label encoder
joblib.dump(le, "/content/drive/MyDrive/label_encoder2.pkl")

['/content/drive/MyDrive/label_encoder2.pkl']